# RiskPredictor — Notebook de entrenamiento (Fase 2)

**Objetivo**: predecir el riesgo (0–100) por zona y hora en Lima, y exportar `predictor_v1.joblib` a `../src/models/`.

## Decisiones de arquitectura (ver memoria del proyecto)
- **XGBoost unificado** (un solo modelo) en vez de `Random Forest + Prophet` (2 + N modelos).
  Las *lag features* le dan la memoria temporal que aportaba Prophet, y un solo modelo
  **comparte fuerza estadística entre zonas** (las zonas con pocos datos se apoyan en el patrón global).
- **`objective='count:poisson'`**: el target es un CONTEO de incidentes por bin (0,1,2,...),
  NO un valor continuo. Poisson es el modelo correcto para datos de conteo.

## Los 3 problemas difíciles que este notebook resuelve (y que la teoría esconde)
1. **Target engineering**: el dataset crudo NO trae una columna "risk_score". Hay que construirla.
2. **Panel completo (los ceros)**: el crudo solo tiene filas donde HUBO crimen. Sin los bins
   en 0, el modelo nunca aprende dónde/cuándo NO pasa nada.
3. **Split temporal**: partir al azar mete el futuro en el train → fuga de datos → métricas mentirosas.

> ⚠️ Datos **sintéticos** para que el pipeline corra end-to-end. Reemplazar por datos abiertos reales.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_poisson_deviance

RNG = np.random.default_rng(7)
pd.set_option('display.max_columns', None)
print('Setup OK — xgboost', xgb.__version__)

## 1. Incidentes crudos sintéticos

Generamos incidentes INDIVIDUALES (como vendría un dataset real: un evento por fila, con
fecha/hora y coordenadas). Inyectamos señal aprendible:
- **Hotspots espaciales**: algunas zonas concentran más crimen (peso relativo).
- **Sesgo nocturno**: más incidentes 20:00–03:00.
- **Sesgo de fin de semana**: viernes/sábado más peligrosos.


In [ ]:
# --- Parametros de simulacion ---
N_DAYS = 90                          # 3 meses de historia
START = pd.Timestamp('2026-01-01')   # fecha fija -> reproducible
N_INCIDENTS = 9000                   # incidentes individuales (crudos)

# Hotspots: (lat, lng, peso_relativo, nombre). Mas peso = mas crimen concentrado.
HOTSPOTS = [
    (-12.066, -77.030, 3.0, 'La Victoria'),
    (-11.985, -77.006, 2.5, 'San Juan de Lurigancho'),
    (-12.121, -77.030, 1.5, 'Miraflores'),
    (-12.056, -77.118, 2.0, 'Callao'),
    (-11.949, -77.061, 1.8, 'Comas'),
    (-12.135, -76.994, 1.0, 'Surco'),
]
INCIDENT_TYPES = ['ROBBERY', 'ASSAULT', 'THEFT', 'VANDALISM']

def hour_weights():
    # Sesgo nocturno: noche x3, madrugada/mañana tranquila x0.5
    w = np.ones(24)
    for h in range(24):
        if h >= 20 or h <= 3:
            w[h] = 3.0
        elif 4 <= h <= 9:
            w[h] = 0.5
    return w / w.sum()

def dow_weights():
    # 0=lunes ... 6=domingo ; vie/sab mas peligrosos
    w = np.array([1.0, 1.0, 1.0, 1.2, 1.8, 2.0, 1.4])
    return w / w.sum()

HW, DW = hour_weights(), dow_weights()
hot_p = np.array([w for *_, w, _ in HOTSPOTS]); hot_p = hot_p / hot_p.sum()

# Probabilidad de cada DIA segun su dia de semana (inyecta sesgo de fin de semana)
day_dows = np.array([(START + pd.Timedelta(days=d)).dayofweek for d in range(N_DAYS)])
day_p = DW[day_dows]; day_p = day_p / day_p.sum()

rows = []
for _ in range(N_INCIDENTS):
    hi = int(RNG.choice(len(HOTSPOTS), p=hot_p))
    lat0, lng0, _, _ = HOTSPOTS[hi]
    d = int(RNG.choice(N_DAYS, p=day_p))
    ts_day = (START + pd.Timedelta(days=d)).normalize()
    rows.append({
        'date': ts_day,
        'hour': int(RNG.choice(range(24), p=HW)),
        'day_of_week': int(ts_day.dayofweek),
        'lat': float(lat0 + RNG.normal(0, 0.004)),   # dispersion ~400m alrededor del hotspot
        'lng': float(lng0 + RNG.normal(0, 0.004)),
        'incident_type': str(RNG.choice(INCIDENT_TYPES)),
    })

raw = pd.DataFrame(rows)
print('Incidentes crudos:', raw.shape)
raw.head()

## 2. Binning espacial (grilla / tiles)

Discretizamos el espacio en celdas cuadradas. Cada incidente cae en una `zone_id`.

> 🔑 **Detalle de integración**: en producción el tamaño de tile DEBE coincidir con el que usa el
> *threshold engine* del API para agrupar reportes (~0.003° ≈ 330m). Si no coinciden, predecís
> riesgo sobre zonas que no matchean con cómo agrupás los reportes reales. Acá usamos `TILE=0.03`
> (~1.1km) solo para que la grilla sintética sea chica y manejable.


In [ ]:
LAT_MIN, LNG_MIN = -12.30, -77.20   # esquina inferior de la bbox de Lima
TILE = 0.03                          # ~3.3km (DEMO). En prod: alinear con tiles del API.

def to_zone(lat, lng):
    gi = int(np.floor((lat - LAT_MIN) / TILE))
    gj = int(np.floor((lng - LNG_MIN) / TILE))
    return gi, gj

raw[['gi', 'gj']] = raw.apply(lambda r: pd.Series(to_zone(r['lat'], r['lng'])), axis=1)
raw['zone_id'] = raw['gi'].astype(str) + '_' + raw['gj'].astype(str)
# Centro del tile = feature espacial ESTABLE (no usamos el lat/lng exacto del incidente)
raw['zone_lat'] = LAT_MIN + (raw['gi'] + 0.5) * TILE
raw['zone_lng'] = LNG_MIN + (raw['gj'] + 0.5) * TILE

print('Zonas con actividad:', raw['zone_id'].nunique())
raw[['zone_id', 'gi', 'gj', 'zone_lat', 'zone_lng']].drop_duplicates().head()

## 3. Panel completo: zona × día × hora (incluyendo los CEROS)

**El gotcha #1.** El crudo solo tiene filas donde hubo crimen. Materializamos el producto
cartesiano completo y rellenamos `count=0` donde no pasó nada. Ese `count` es el TARGET.


In [ ]:
zones = raw[['zone_id', 'gi', 'gj', 'zone_lat', 'zone_lng']].drop_duplicates()
dates = pd.date_range(START, periods=N_DAYS, freq='D')
hours = list(range(24))

# Producto cartesiano zona x fecha x hora
grid = (zones.assign(key=1)
        .merge(pd.DataFrame({'date': dates, 'key': 1}), on='key')
        .merge(pd.DataFrame({'hour': hours, 'key': 1}), on='key')
        .drop(columns='key'))

# Conteo real por bin
counts = (raw.groupby(['zone_id', 'date', 'hour']).size()
          .rename('count').reset_index())

panel = grid.merge(counts, on=['zone_id', 'date', 'hour'], how='left')
panel['count'] = panel['count'].fillna(0).astype(int)

# Timestamp e indice temporal (clave para los lags y el split)
panel['ts'] = panel['date'] + pd.to_timedelta(panel['hour'], unit='h')
panel['day_of_week'] = panel['ts'].dt.dayofweek
panel = panel.sort_values(['zone_id', 'ts']).reset_index(drop=True)

print('Panel:', panel.shape, '| % bins con incidentes:', round((panel['count'] > 0).mean() * 100, 2), '%')
panel.head()

## 4. Feature engineering espacio-temporal

Las **lag features** le dan a XGBoost la memoria que aportaba Prophet:
- `inc_last_24h` / `inc_last_7d`: actividad reciente de la propia zona.
- `inc_same_hour_7d`: misma franja horaria en días previos (estacionalidad diaria).
- `inc_adj_24h`: actividad de zonas VECINAS (el "barrio de al lado está caliente").

> 🔑 Todas usan `shift(1)` para **excluir el bin actual** → evita que el modelo se mire a sí mismo
> (fuga de datos). Las primeras observaciones sin historia se rellenan con 0 (cold start).


In [ ]:
# Ciclico (hora y dia de semana)
panel['hour_sin'] = np.sin(2*np.pi*panel['hour']/24)
panel['hour_cos'] = np.cos(2*np.pi*panel['hour']/24)
panel['dow_sin']  = np.sin(2*np.pi*panel['day_of_week']/7)
panel['dow_cos']  = np.cos(2*np.pi*panel['day_of_week']/7)
panel['is_weekend'] = (panel['day_of_week'] >= 5).astype(int)

# Lags por zona (shift(1) excluye el bin actual)
g = panel.groupby('zone_id', group_keys=False)
panel['inc_last_24h'] = g['count'].apply(lambda s: s.shift(1).rolling(24, min_periods=1).sum())
panel['inc_last_7d']  = g['count'].apply(lambda s: s.shift(1).rolling(24*7, min_periods=1).sum())

# Mismo horario, ultimos 7 dias (por zona+hora, en orden de fecha)
panel['inc_same_hour_7d'] = (panel.groupby(['zone_id', 'hour'], group_keys=False)['count']
                             .apply(lambda s: s.shift(1).rolling(7, min_periods=1).mean()))

# Contexto espacial: vecinos = tiles adyacentes (gi,gj +-1)
zinfo = panel[['zone_id', 'gi', 'gj']].drop_duplicates().set_index('zone_id')
gi_gj_to_zone = {(r.gi, r.gj): z for z, r in zinfo.iterrows()}
neighbors = {}
for z, r in zinfo.iterrows():
    nb = []
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue
            key = (r.gi + di, r.gj + dj)
            if key in gi_gj_to_zone:
                nb.append(gi_gj_to_zone[key])
    neighbors[z] = nb

# Suma de inc_last_24h de los vecinos, alineada por timestamp
pivot24 = panel.pivot_table(index='ts', columns='zone_id', values='inc_last_24h', fill_value=0)
adj_series = {z: (pivot24[neighbors[z]].sum(axis=1) if neighbors[z]
                  else pd.Series(0.0, index=pivot24.index)) for z in neighbors}
panel['inc_adj_24h'] = panel.apply(lambda r: float(adj_series[r['zone_id']].get(r['ts'], 0.0)), axis=1)

# Cold start -> 0
lag_cols = ['inc_last_24h', 'inc_last_7d', 'inc_same_hour_7d', 'inc_adj_24h']
panel[lag_cols] = panel[lag_cols].fillna(0)

# Encoding de zona para el modelo (XGBoost necesita numerico)
zone_cat = panel['zone_id'].astype('category')
panel['zone_code'] = zone_cat.cat.codes
ZONE_TO_CODE = {z: i for i, z in enumerate(zone_cat.cat.categories)}

print(panel[lag_cols].describe().round(2))
panel.head()

## 5. Split TEMPORAL + entrenamiento

**El gotcha #2.** NO usamos `train_test_split` aleatorio: eso mete fechas futuras en el train y
contamina los lags. Entrenamos con las fechas viejas, validamos con las nuevas.


In [ ]:
cutoff = dates[int(N_DAYS * 0.8)]   # 80% historia para train

FEATURES = ['zone_code', 'zone_lat', 'zone_lng',
            'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend',
            'inc_last_24h', 'inc_last_7d', 'inc_same_hour_7d', 'inc_adj_24h']
TARGET = 'count'

train = panel[panel['date'] < cutoff]
test  = panel[panel['date'] >= cutoff]
print('train:', train.shape, '| test:', test.shape, '| cutoff:', cutoff.date())

model = xgb.XGBRegressor(
    objective='count:poisson',   # target = CONTEO -> Poisson (no regresion comun)
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=5,
    random_state=7,
    n_jobs=-1,
)
model.fit(train[FEATURES], train[TARGET],
          eval_set=[(test[FEATURES], test[TARGET])], verbose=False)
print('Entrenado.')

## 6. Evaluación

`MAE`/`RMSE` sobre el conteo + **Poisson deviance** (la métrica propia de conteos).
Y la importancia de features: esperamos que los lags y el ciclo horario pesen.


In [ ]:
pred = np.clip(model.predict(test[FEATURES]), 0, None)   # lambda Poisson >= 0
print('MAE :', round(mean_absolute_error(test[TARGET], pred), 4))
print('RMSE:', round(mean_squared_error(test[TARGET], pred) ** 0.5, 4))
print('Poisson deviance:', round(mean_poisson_deviance(test[TARGET], np.clip(pred, 1e-6, None)), 4))

imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('\nImportancia de features:')
print(imp.round(4))

## 7. Target engineering: de λ (conteo esperado) a `risk_score` 0–100

**El gotcha #3.** El modelo predice un CONTEO esperado (λ). El `risk_score` que pide la API es una
**decisión de producto**: ¿qué conteo equivale a "riesgo máximo (100)"? Usamos el percentil 99 del
train como tope (`LAMBDA_CAP`) y escalamos lineal. Ajustable según cómo querás que se vea el mapa.


In [ ]:
LAMBDA_CAP = float(max(0.5, np.quantile(model.predict(train[FEATURES]), 0.99)))
print('LAMBDA_CAP (p99 del train):', round(LAMBDA_CAP, 3))

def to_risk_score(lmbda):
    return int(np.clip(round(100 * lmbda / LAMBDA_CAP), 0, 100))

test = test.copy()
test['lambda'] = pred
test['risk_score'] = test['lambda'].map(to_risk_score)

print('\nTop 10 bins de mayor riesgo predicho:')
test[['zone_id', 'date', 'hour', 'day_of_week', 'count', 'lambda', 'risk_score']] \
    .sort_values('risk_score', ascending=False).head(10)

## 8. Función de inferencia (contrato `PredictRiskResponse`)

> 🔑 En producción las lag features **no se calculan acá**: vienen de consultas a la BD (incidentes
> recientes por zona) o de Redis. Acá las pasamos explícitas (`lags`) para demostrar el flujo.
> El resultado de este modelo se cachea en la tabla `risk_zones` (batch / cron), no se corre por request.


In [ ]:
def predict_risk(zone_id, hour, day_of_week, lags, bundle):
    code = bundle['zone_to_code'].get(zone_id, -1)
    zlat, zlng = bundle['zone_centers'].get(zone_id, (0.0, 0.0))
    row = pd.DataFrame([{
        'zone_code': code, 'zone_lat': zlat, 'zone_lng': zlng,
        'hour_sin': np.sin(2*np.pi*hour/24),  'hour_cos': np.cos(2*np.pi*hour/24),
        'dow_sin':  np.sin(2*np.pi*day_of_week/7), 'dow_cos': np.cos(2*np.pi*day_of_week/7),
        'is_weekend': int(day_of_week >= 5),
        'inc_last_24h':     lags.get('inc_last_24h', 0),
        'inc_last_7d':      lags.get('inc_last_7d', 0),
        'inc_same_hour_7d': lags.get('inc_same_hour_7d', 0),
        'inc_adj_24h':      lags.get('inc_adj_24h', 0),
    }])[bundle['features']]
    lmbda = float(np.clip(bundle['model'].predict(row)[0], 0, None))
    cap = bundle['lambda_cap']
    return {
        'risk_score': int(np.clip(round(100 * lmbda / cap), 0, 100)),
        'predicted_hour': hour,
        'confidence': round(float(min(1.0, lmbda / cap)), 3),   # proxy: calibrar en Fase 2.5
        'expected_count': round(lmbda, 3),
    }

print('definido')

## 9. Exportar artefacto

Bundle (modelo + metadata de grilla + encoders) → `../src/models/predictor_v1.joblib`.
`RiskPredictor.load_model()` lo levanta con `joblib.load()`.


In [ ]:
HERE = Path.cwd()
ML_ROOT = HERE.parent if HERE.name == 'notebooks' else HERE   # corre desde ml/notebooks o ml/
MODELS_DIR = ML_ROOT / 'src' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT = MODELS_DIR / 'predictor_v1.joblib'

zc = panel[['zone_id', 'zone_lat', 'zone_lng']].drop_duplicates().set_index('zone_id')
bundle = {
    'version': 'v1',
    'model': model,
    'features': FEATURES,
    'lambda_cap': LAMBDA_CAP,
    'tile': TILE, 'lat_min': LAT_MIN, 'lng_min': LNG_MIN,
    'zone_to_code': ZONE_TO_CODE,
    'zone_centers': {z: (float(r.zone_lat), float(r.zone_lng)) for z, r in zc.iterrows()},
}
joblib.dump(bundle, ARTIFACT)
print('Guardado:', ARTIFACT.resolve())

# Smoke test del artefacto serializado
loaded = joblib.load(ARTIFACT)
demo_zone = panel['zone_id'].mode()[0]   # zona mas activa
print('Viernes 23h (zona activa) ->',
      predict_risk(demo_zone, 23, 4, {'inc_last_24h':3,'inc_last_7d':18,'inc_same_hour_7d':1.2,'inc_adj_24h':5}, loaded))
print('Martes  07h (zona activa) ->',
      predict_risk(demo_zone, 7, 1, {'inc_last_24h':0,'inc_last_7d':2,'inc_same_hour_7d':0,'inc_adj_24h':0}, loaded))

## Próximos pasos

1. **Datos reales** de Lima (datos abiertos georreferenciados) → re-entrenar. Re-evaluar `LAMBDA_CAP`.
2. **Alinear `TILE`** con el tamaño de tile real del threshold engine del API (~0.003°).
3. **Job batch (cron)** que corra el predictor y cachee `risk_score` por zona/hora en la tabla `risk_zones`.
4. La capa `infrastructure` debe **calcular los lags desde la BD** (no a mano como acá).
5. Wirear router `/predict` en `main.py` y exponer al mapa (lectura desde `risk_zones`, < 2s).
6. (Opcional) Prophet solo para 3–5 zonas top, para estacionalidad quincena/feriados que XGBoost no capta sin features explícitos.
